# 4.03 Grid Search de Arboles Azarosos — Opcion B: validacion local

Recorre la grilla de hiperparametros < feature_fraction, minsplit, minbucket, maxdepth > (con **cp fijo en -1**), entrenando para cada combinacion el ensemble arbol por arbol (igual que en `z420`), separando el 202107 en train/validacion (70/30, particion estratificada) y midiendo la ganancia localmente en cada punto de `PARAM$grabar` (por default `1, 2, 4, 8, 16, 32`), con la misma formula de `z290_TareaHogar_02` (estimulo 975000 / costo -25000, corte en prob > 0.025).

<br>No consume submits de Kaggle, asi que esta grilla puede ser mucho mas grande que la de la Opcion A — el costo es tiempo de computo.

<br>Al final se muestra un ranking de combinaciones por la ganancia del ensemble completo (ultimo punto de `PARAM$grabar`). Elegis la mejor y la cargas manualmente en `z420` (los campos `PARAM$feature_fraction` y `PARAM$rpart`) para el entrenamiento final y el submit a Kaggle.

#### Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

---

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Aug 23 02:14:48 AM 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671270,35.9,1473300,78.7,1473300,78.7
Vcells,1242666,9.5,8388608,64.0,1978712,15.1


In [3]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart



Aqui debe cargar SU semilla primigenia, y ajustar la grilla de hiperparametros a explorar

In [17]:
PARAM <- list()
PARAM$semilla_primigenia <- 346321
PARAM$training_pct <- 70L # split train/validacion DENTRO de 202107, no toca Kaggle

PARAM$rpart$cp <- -1 # fijo, no se barre en este grid search

PARAM$num_trees_max <- 16 # arboles por combinacion, igual que z420

# puntos del ensemble donde se mide la ganancia, igual que en z420
PARAM$grabar <- c(1, 2, 4, 8, 16)#, 32)

# grilla de hiperparametros a explorar
#  al ser validacion local (no consume submits de Kaggle) esta grilla puede ser
#  mucho mas grande que la de la Opcion A, el unico costo es tiempo de computo
PARAM$grid$feature_fraction <- c(0.5, 0.7)
PARAM$grid$minsplit  <- c(1000, 400, 100, 20)
PARAM$grid$minbucket <- c(5, 20, 50)
PARAM$grid$maxdepth  <- c(6, 10, 14)

In [18]:
PARAM

$semilla_primigenia
[1] 346321

$training_pct
[1] 70

$rpart
$rpart$cp
[1] -1


$num_trees_max
[1] 16

$grabar
[1]  1  2  4  8 16

$grid
$grid$feature_fraction
[1] 0.5 0.7

$grid$minsplit
[1] 1000  400  100   20

$grid$minbucket
[1]  5 20 50

$grid$maxdepth
[1]  6 10 14

### Preview: dimensiono el trabajo antes de correr el grid search

In [19]:
# --- conteo de combinaciones antes de correr, para dimensionar el trabajo ---
library(parallel)
n_cores <- detectCores()
cat("Nucleos disponibles:", n_cores, "\n\n")

grid_preview <- CJ(
  maxdepth = PARAM$grid$maxdepth,
  minbucket = PARAM$grid$minbucket,
  minsplit = PARAM$grid$minsplit,
  feature_fraction = PARAM$grid$feature_fraction
)

total_bruto <- nrow(grid_preview)
grid_preview_filtrado <- grid_preview[minbucket <= minsplit]
total_filtrado <- nrow(grid_preview_filtrado)

cat("Combinaciones totales (sin filtrar):", total_bruto, "\n")
cat("Combinaciones validas (minbucket <= minsplit):", total_filtrado, "\n")
cat("Combinaciones descartadas por el filtro:", total_bruto - total_filtrado, "\n")
cat("Arboles a entrenar en total:", total_filtrado * PARAM$num_trees_max, "\n")


Nucleos disponibles: 2 

Combinaciones totales (sin filtrar): 72 
Combinaciones validas (minbucket <= minsplit): 66 
Combinaciones descartadas por el filtro: 6 
Arboles a entrenar en total: 1056 


In [20]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp430"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [21]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

# me quedo solo con los datos que tienen clase conocida, es decir 202107
#  (202109 tiene clase_ternaria vacia, no sirve para validar localmente)
dataset <- dataset[clase_ternaria != ""]

### Particion train / validacion local (70/30 estratificada por clase_ternaria)

In [22]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa
particionar <- function(data, division, agrupa = "", campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by = agrupa
  ]
}

# hago la particion UNA sola vez, con semilla fija, para que todas las
#  combinaciones de la grilla se comparen sobre el mismo train/validation
particionar(dataset,
  division= c(PARAM$training_pct, 100L - PARAM$training_pct),
  agrupa= "clase_ternaria",
  seed= PARAM$semilla_primigenia
)

dtr  <- dataset[fold == 1] # 70% training
dval <- dataset[fold == 2] # 30% validacion local

campos_buenos <- copy(setdiff(colnames(dtr), c("clase_ternaria", "fold")))

### Funcion que entrena el ensemble y devuelve la ganancia en validacion

In [23]:
# entrena el ensemble de PARAM$num_trees_max arboles para una combinacion de
#  hiperparametros dada, y devuelve la ganancia normalizada sobre dval en cada
#  punto de PARAM$grabar (una fila por punto, columnas arbolito y ganancia)
ArbolesAzarososGanancia <- function(feature_fraction, rpart_control) {

  # misma semilla para todas las combinaciones, asi la unica diferencia entre
  #  combinaciones es el hiperparametro, no el azar
  set.seed(PARAM$semilla_primigenia)

  tb_pred <- dval[, list(numero_de_cliente, clase_ternaria)]
  tb_pred[, prob_acumulada := 0]

  resultados <- data.table(arbolito = integer(), ganancia = numeric())

  for (arbolito in seq(PARAM$num_trees_max)) {
    qty_campos_a_utilizar <- as.integer(length(campos_buenos) * feature_fraction)
    campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
    campos_random <- paste(campos_random, collapse= " + ")
    formulita <- paste0("clase_ternaria ~ ", campos_random)

    modelo <- rpart(formulita, data= dtr, xval= 0, control= rpart_control)
    prediccion <- predict(modelo, dval, type= "prob")
    tb_pred[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]

    if (!(arbolito %in% PARAM$grabar)) next

    # umbral sobre la SUMA acumulada, equivalente a promedio > 1/40
    umbral_corte <- arbolito / 40
    tb_pred[, Predicted := prob_acumulada > umbral_corte]

    ganancia_test <- tb_pred[, sum(ifelse(Predicted,
        ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
        0))]

    # escalo la ganancia como si fuera todo el dataset (misma logica que z290)
    ganancia_test_normalizada <- ganancia_test / ((100 - PARAM$training_pct) / 100)

    resultados <- rbindlist(list( resultados,
      data.table(arbolito= arbolito, ganancia= ganancia_test_normalizada)
    ))
  }

  resultados
}

In [16]:
rpart_control_rapido <- list(
  cp = PARAM$rpart$cp,
  minsplit = 200,
  minbucket = 20,
  maxdepth = 10,
  maxcompete = 0,
  maxsurrogate = 0
)

t0 <- Sys.time()
resultado <- ArbolesAzarososGanancia(0.7, rpart_control_rapido)
t1 <- Sys.time()
print(t1 - t0)

Time difference of 20.03932 mins


In [15]:
dim(dtr)
length(campos_buenos)

[1] 164479    156

[1] 154

### Grid Search

In [24]:
# archivo donde se guarda el checkpoint del grid search (que combinaciones ya se corrieron)
archivo_grid <- "gridsearch_localNuevasCombinaciones.txt"

if (file.exists(archivo_grid)) {
  tb_grid <- fread(archivo_grid)
} else {
  tb_grid <- data.table(
    combo_id = integer(),
    feature_fraction = numeric(),
    cp = numeric(),
    minsplit = integer(),
    minbucket = integer(),
    maxdepth = integer(),
    arbolito = integer(),
    ganancia = numeric()
  )
}

# --- armo la grilla completa, mismo orden que los for anidados originales ---
grid <- CJ(
  maxdepth = PARAM$grid$maxdepth,
  minbucket = PARAM$grid$minbucket,
  minsplit = PARAM$grid$minsplit,
  feature_fraction = PARAM$grid$feature_fraction,
  sorted = FALSE
)
setcolorder(grid, c("feature_fraction", "minsplit", "minbucket", "maxdepth"))
setorder(grid, feature_fraction, minsplit, minbucket, maxdepth)

grid <- grid[minbucket <= minsplit]  # mismo filtro que ya tenias
grid[, combo_id := .I]

cat("Combinaciones a procesar:", nrow(grid), "\n")

# --- filtro las combinaciones que ya tienen TODOS los puntos de grabar calculados ---
grid[, ya_completa := mapply(function(ff, ms, mb, md) {
  puntos_hechos <- tb_grid[
    feature_fraction == ff & minsplit == ms & minbucket == mb & maxdepth == md,
    arbolito
  ]
  all(PARAM$grabar %in% puntos_hechos)
}, feature_fraction, minsplit, minbucket, maxdepth)]

grid_pendiente <- grid[ya_completa == FALSE]
cat("Combinaciones pendientes:", nrow(grid_pendiente), "\n")

# --- proceso en tandas, grabando checkpoint despues de cada una ---
tam_tanda <- 4  # chico: 2 nucleos + riesgo de desconexion en Colab
tandas <- split(seq_len(nrow(grid_pendiente)), ceiling(seq_len(nrow(grid_pendiente)) / tam_tanda))

for (t in tandas) {

  sub_grid <- grid_pendiente[t]
  cat("\nProcesando combo_id:", sub_grid$combo_id, "\n")
  flush.console()

  resultados <- mcmapply(
    function(ff, ms, mb, md, cid) {
      rpart_control <- list(
        cp = PARAM$rpart$cp,
        minsplit = ms,
        minbucket = mb,
        maxdepth = md,
        maxcompete = 0,
        maxsurrogate = 0
      )
      res <- ArbolesAzarososGanancia(ff, rpart_control)

      data.table(
        combo_id = cid,
        feature_fraction = ff,
        cp = PARAM$rpart$cp,
        minsplit = ms,
        minbucket = mb,
        maxdepth = md,
        arbolito = res$arbolito,
        ganancia = res$ganancia
      )
    },
    sub_grid$feature_fraction, sub_grid$minsplit, sub_grid$minbucket,
    sub_grid$maxdepth, sub_grid$combo_id,
    SIMPLIFY = FALSE,
    mc.cores = n_cores
  )

  tb_nueva <- rbindlist(resultados)
  tb_grid <- rbindlist(list(tb_grid, tb_nueva))

  fwrite(tb_grid, file = archivo_grid, sep = "\t")
}

cat("\n\nGrid search completo. Total de filas en tb_grid:", nrow(tb_grid), "\n")


Combinaciones a procesar: 66 
Combinaciones pendientes: 66 

Procesando combo_id: 1 2 3 4 

Procesando combo_id: 5 6 7 8 

Procesando combo_id: 9 10 11 12 

Procesando combo_id: 13 14 15 16 

Procesando combo_id: 17 18 19 20 

Procesando combo_id: 21 22 23 24 

Procesando combo_id: 25 26 27 28 

Procesando combo_id: 29 30 31 32 

Procesando combo_id: 33 34 35 36 

Procesando combo_id: 37 38 39 40 

Procesando combo_id: 41 42 43 44 

Procesando combo_id: 45 46 47 48 

Procesando combo_id: 49 50 51 52 

Procesando combo_id: 53 54 55 56 

Procesando combo_id: 57 58 59 60 

Procesando combo_id: 61 62 63 64 

Procesando combo_id: 65 66 


Grid search completo. Total de filas en tb_grid: 330 


### Resultado: ranking de combinaciones por ganancia del ensemble completo
Cada combinacion tiene una fila por cada punto de `PARAM$grabar`, para ver como evoluciona la ganancia a medida que se agregan arboles. El ranking final se hace sobre el ultimo punto (ensemble completo).

In [26]:
# ranking final: ganancia del ensemble COMPLETO (ultimo punto de PARAM$grabar) por combinacion
tb_final <- tb_grid[arbolito == max(PARAM$grabar)]
setorder(tb_final, -ganancia)
tb_final[1:20]

combo_id,feature_fraction,cp,minsplit,minbucket,maxdepth,arbolito,ganancia
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>
32,0.5,-1,1000,50,10,16,493500000
6,0.5,-1,20,20,14,16,490333333
33,0.5,-1,1000,50,14,16,488583333
55,0.7,-1,400,50,6,16,486083333
12,0.5,-1,100,20,14,16,484833333
65,0.7,-1,1000,50,10,16,484750000
15,0.5,-1,100,50,14,16,483083333
46,0.7,-1,100,50,6,16,482500000
26,0.5,-1,1000,5,10,16,482166667


In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Fri Aug 21 04:29:44 PM 2026"